In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers

In [ ]:
from threading import Thread

import torch
import gradio as gr
from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
SYSTEM_PROMPT = """
You are a Synthetic Data Generator.
Generate realistic, diverse, and internally consistent synthetic datasets based on the user's requirements.

Rules:
- Follow the requested schema, fields, data types, constraints, and number of records.
- Generate only fictional data; never use real people's private information.
- Keep related fields logically consistent.
- Respect specified ranges, distributions, and formats.
- Return only the dataset, without explanations.
- If JSON is requested, return valid JSON. If CSV is requested, return valid CSV.
- If requirements are unclear, ask a concise clarification question.
- Verify the data before returning it.
"""

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config,
    )
    model.eval()
    return tokenizer, model

In [ ]:
tokenizer, model = load_model(LLAMA)

In [ ]:
def extract_text(content):
    """content can be a plain string, or Gradio's list-of-parts format:
    [{'text': '...', 'type': 'text'}, ...]. Normalize to a single string."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict) and part.get("type") == "text":
                parts.append(part.get("text", ""))
            elif isinstance(part, str):
                parts.append(part)
        return "\n".join(p for p in parts if p)
    return ""

In [ ]:
def build_messages(message, history):
    """Works with legacy tuple history and both flavors of Gradio 'messages' history
    (plain-string content and the newer list-of-parts content)."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for turn in history or []:
        if isinstance(turn, dict):                       # type="messages"
            text = extract_text(turn.get("content"))
            if text.strip():
                messages.append({"role": turn["role"], "content": text})
        else:                                            # legacy [user, assistant] pairs
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": extract_text(user_msg)})
            if bot_msg:
                messages.append({"role": "assistant", "content": extract_text(bot_msg)})

    messages.append({"role": "user", "content": message})
    return messages

In [ ]:
def chat(message, history, max_tokens=2000):
    messages = build_messages(message, history)

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    partial = ""
    for token in streamer:
        partial += token
        yield partial          # FIX: only the new text, not prompt + history echoed back

    thread.join()

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    title="Synthetic Data Generator",
    description="Describe the schema and number of records you need. Ask for JSON or CSV.",
    examples=[
        "Generate 10 records as JSON: id, full_name, email, signup_date (2024), plan (free/pro/enterprise), monthly_spend_usd.",
        "Give me a CSV of 25 e-commerce orders: order_id, customer_id, product, quantity (1-5), unit_price, order_date, status.",
        "Create 15 employee records in JSON: emp_id, name, department, joining_date, salary_lpa between 4 and 25, manager_id.",
    ],
).launch(debug=True)